In [1]:
import pandas as pd
import os

# Check if the files exist in this folder
print("Checking file locations...")
if os.path.exists('diagnoses_icd.csv'):
    print("✅ diagnoses_icd file found!")
else:
    print("❌ diagnoses_icd file NOT found. Check the folder/file name.")

if os.path.exists('discharge.csv'):
    print("✅ discharge file found!")
else:
    print("❌ discharge file NOT found. Make sure the file is next to this .ipynb file.")

Checking file locations...
✅ diagnoses_icd file found!
✅ discharge file found!


In [3]:
# Filter Lung Cancer Patients (Cohort Selection)
# This replaces the SQL WHERE process to search for C34/D02 codes.
print("Processing diagnosis data... (please wait)")

# 1. Read diagnosis data
# compression='gzip' means we read the .gz file directly without extracting
df_diag = pd.read_csv('diagnoses_icd.csv')

# 2. Filter ICD 10 Codes (C34.* or D02.2*)
lung_cancer_cohort = df_diag[
    df_diag['icd_code'].str.startswith(('C34', 'D022')) 
]

# Get unique Subject IDs only
unique_patients = lung_cancer_cohort['subject_id'].nunique()
print(f"Done! Found {len(lung_cancer_cohort)} lung cancer diagnoses.")
print(f"Number of unique patients: {unique_patients}")

Processing diagnosis data... (please wait)
Done! Found 4286 lung cancer diagnoses.
Number of unique patients: 2294


In [4]:
# Merge with Medical Records (Join Text)
print("Reading medical notes (this is a large file, may take 1-2 minutes)...")

# 1. Read notes file
# We only keep essential columns to save RAM
cols_to_keep = ['subject_id', 'hadm_id', 'note_id', 'text']
df_notes = pd.read_csv('discharge.csv', usecols=cols_to_keep)

# 2. MERGE (INNER JOIN)
# Only keep notes belonging to patients in the lung_cancer_cohort
final_df = pd.merge(
    lung_cancer_cohort[['subject_id', 'hadm_id', 'icd_code']], # Left Table
    df_notes, # Right Table
    on=['subject_id', 'hadm_id'],
    how='inner'
)

print(f"SUCCESS: Data merge complete.")
print(f"Total records ready for processing: {len(final_df)}")

# Display a preview
display(final_df.head(3))

Reading medical notes (this is a large file, may take 1-2 minutes)...
SUCCESS: Data merge complete.
Total records ready for processing: 2333


,subject_id,hadm_id,icd_code,note_id,text
0,10002348,22725460,C3490,10002348-DS-13,\nName: ___ Unit No: ___...
1,10003299,20940957,C342,10003299-DS-9,\nName: ___ Unit No: ___\n...
2,10003299,27373340,C342,10003299-DS-10,\nName: ___ Unit No: ___\n...


In [13]:
print("--- STEP 3: TEXT PRE-PROCESSING (TIERED LOGIC OPTIMIZED) ---")

def extract_relevant_section(text):
    """
    Extracts sections relevant to Lung Cancer Histology based on WHO 2021 Tiered Logic.
    Ensures subtypes and pattern percentages are captured.
    """
    if not isinstance(text, str):
        return ""
        
    text_lower = text.lower()
    
    # LEVEL 1 & 2 Keywords (Primary Classes)
    primary_keywords = [
        "pathology", "diagnosis", "microscopic", "tumor", "tumour", "malignancy",
        "adenocarcinoma", "squamous", "neuroendocrine", "carcinoma", 
        "small cell", "large cell"
    ]
    
    # LEVEL 3 Keywords (Adeno Subtypes & Patterns)
    adeno_keywords = [
        "lepidic", "acinar", "papillary", "micropapillary", "solid", "cribriform",
        "mucinous", "colloid", "fetal", "enteric", "situ", "minimally invasive", "ais", "mia"
    ]
    
    # LEVEL 4, 5, 6 Keywords (Squamous & NE Variants)
    variant_keywords = [
        "keratinizing", "basaloid", "lymphoepithelial", "typical", "atypical", "carcinoid"
    ]
    
    # Combine all keywords
    all_keywords = primary_keywords + adeno_keywords + variant_keywords
    
    extracted_parts = []
    
    # Strategy: Capture context around keywords
    # We use a wider window (600 chars before, 3000 after) to catch full descriptions/percentages
    for key in all_keywords:
        start_idx = text_lower.find(key)
        if start_idx != -1:
            s_pos = max(0, start_idx - 600)
            e_pos = min(len(text), start_idx + 3000)
            
            chunk = text[s_pos:e_pos]
            extracted_parts.append(f"--- CONTEXT: {key.upper()} ---\n...{chunk}...")
            
    if not extracted_parts:
        return "NO_KEYWORDS_FOUND (Taking top section):\n" + text[:3000]
    
    # Join parts and remove exact duplicates implicitly via logic if needed, 
    # but strictly simply joining is safer for context preservation.
    return "\n\n".join(extracted_parts)

# Apply the function
print("Processing text with Tiered Logic extraction...")
final_df['processed_text'] = final_df['text'].apply(extract_relevant_section)

# Save to CSV
output_filename = 'mimic_lung_cancer_tiered_ready.csv'
final_df[['subject_id', 'hadm_id', 'processed_text']].to_csv(output_filename, index=False)

print(f"SUCCESS: Processed data saved to '{output_filename}'")

--- STEP 3: TEXT PRE-PROCESSING (TIERED LOGIC OPTIMIZED) ---
Processing text with Tiered Logic extraction...
SUCCESS: Processed data saved to 'mimic_lung_cancer_tiered_ready.csv'
